# Ingestion – Scrape Website & Store Raw Data

Collect links from course website

In [1]:
import requests
from bs4 import BeautifulSoup
import json
import os
from urllib.parse import urljoin, urlparse

In [2]:
BASE_URL = "https://pantelis.github.io/courses/ai/in-person.html"
DOMAIN = "pantelis.github.io"

visited = set()
collected_links = []

In [3]:
def is_valid(url):
    parsed = urlparse(url)
    return DOMAIN in parsed.netloc

In [4]:
def scrape_page(url):
    if url in visited:
        return
    visited.add(url)

    print("Scraping:", url)

    try:
        resp = requests.get(url, timeout=10)
        soup = BeautifulSoup(resp.text, "html.parser")
    except:
        print("Failed:", url)
        return

    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        full_url = urljoin(url, href)

        # Only collect resource files
        if full_url.endswith((".pdf", ".ppt", ".pptx", ".mp4", ".html")):
            if is_valid(full_url) and full_url not in collected_links:
                collected_links.append(full_url)

        # Recursively follow internal HTML pages 
        if (
            is_valid(full_url)
            and full_url.endswith(".html")
            and full_url not in visited
        ):
            scrape_page(full_url)


In [5]:
scrape_page(BASE_URL)

os.makedirs("raw_data", exist_ok=True)
with open("raw_data/links.json", "w") as f:
    json.dump(collected_links, f, indent=2)

print("\n\nTotal links collected:", len(collected_links))
collected_links[:20]


Scraping: https://pantelis.github.io/courses/ai/in-person.html
Scraping: https://pantelis.github.io/index.html
Scraping: https://pantelis.github.io/book/foundations/index.html
Scraping: https://pantelis.github.io/book/dnn/index.html
Scraping: https://pantelis.github.io/book/2d-perception/index.html
Scraping: https://pantelis.github.io/book/kinematics/index.html
Scraping: https://pantelis.github.io/book/state-estimation/index.html
Scraping: https://pantelis.github.io/book/llm/index.html
Scraping: https://pantelis.github.io/book/multimodal/index.html
Scraping: https://pantelis.github.io/book/task-planning/index.html
Scraping: https://pantelis.github.io/book/global-planning/index.html
Scraping: https://pantelis.github.io/book/local-planning/index.html
Scraping: https://pantelis.github.io/book/mdp/index.html
Scraping: https://pantelis.github.io/book/rl/index.html
Scraping: https://pantelis.github.io/book/vla/index.html
Scraping: https://pantelis.github.io/courses/ai/index.html
Scraping: ht

['https://pantelis.github.io/index.html',
 'https://pantelis.github.io/book/foundations/index.html',
 'https://pantelis.github.io/book/dnn/index.html',
 'https://pantelis.github.io/book/2d-perception/index.html',
 'https://pantelis.github.io/book/kinematics/index.html',
 'https://pantelis.github.io/book/state-estimation/index.html',
 'https://pantelis.github.io/book/llm/index.html',
 'https://pantelis.github.io/book/multimodal/index.html',
 'https://pantelis.github.io/book/task-planning/index.html',
 'https://pantelis.github.io/book/global-planning/index.html',
 'https://pantelis.github.io/book/local-planning/index.html',
 'https://pantelis.github.io/book/mdp/index.html',
 'https://pantelis.github.io/book/rl/index.html',
 'https://pantelis.github.io/book/vla/index.html',
 'https://pantelis.github.io/courses/ai/index.html',
 'https://pantelis.github.io/courses/robotics/index.html',
 'https://pantelis.github.io/courses/cv/index.html',
 'https://pantelis.github.io/about.html',
 'https://p

In [ ]:
urls

['https://pantelis.github.io/index.html',
 'https://pantelis.github.io/book/foundations/index.html',
 'https://pantelis.github.io/book/dnn/index.html',
 'https://pantelis.github.io/book/2d-perception/index.html',
 'https://pantelis.github.io/book/kinematics/index.html',
 'https://pantelis.github.io/book/state-estimation/index.html',
 'https://pantelis.github.io/book/llm/index.html',
 'https://pantelis.github.io/book/multimodal/index.html',
 'https://pantelis.github.io/book/task-planning/index.html',
 'https://pantelis.github.io/book/global-planning/index.html',
 'https://pantelis.github.io/book/local-planning/index.html',
 'https://pantelis.github.io/book/mdp/index.html',
 'https://pantelis.github.io/book/rl/index.html',
 'https://pantelis.github.io/book/vla/index.html',
 'https://pantelis.github.io/courses/ai/index.html',
 'https://pantelis.github.io/courses/robotics/index.html',
 'https://pantelis.github.io/courses/cv/index.html',
 'https://pantelis.github.io/about.html',
 'https://p

Scrape data from collected links

In [ ]:
import json
import requests
from bs4 import BeautifulSoup
import os

def clean_text(text):
    return " ".join(text.split())

# Load list of URLs from links.json
base = os.path.dirname(os.path.abspath("AI_Project"))  
path = os.path.join(base, "raw_data", "links.json")

with open(path, "r") as f:
    urls = json.load(f)

all_content = ""

for url in urls:
    print(f"Scraping: {url}")

    try:
        r = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        soup = BeautifulSoup(r.text, "html.parser")

        # Extract readable text
        for script in soup(["script", "style", "header", "footer", "nav"]):
            script.extract()

        page_text = clean_text(soup.get_text(separator=" "))

        all_content += f"\n\n==== URL: {url} ====\n{page_text}\n"

    except Exception as e:
        print(f"Failed to scrape {url}: {e}")

# Save combined content
with open("scraped_content.txt", "w", encoding="utf-8") as f:
    f.write(all_content)

print("Scraping complete! Output saved to scraped_content.txt")


Scraping: https://pantelis.github.io/index.html
Scraping: https://pantelis.github.io/book/foundations/index.html
Scraping: https://pantelis.github.io/book/dnn/index.html
Scraping: https://pantelis.github.io/book/2d-perception/index.html
Scraping: https://pantelis.github.io/book/kinematics/index.html
Scraping: https://pantelis.github.io/book/state-estimation/index.html
Scraping: https://pantelis.github.io/book/llm/index.html
Scraping: https://pantelis.github.io/book/multimodal/index.html
Scraping: https://pantelis.github.io/book/task-planning/index.html
Scraping: https://pantelis.github.io/book/global-planning/index.html
Scraping: https://pantelis.github.io/book/local-planning/index.html
Scraping: https://pantelis.github.io/book/mdp/index.html
Scraping: https://pantelis.github.io/book/rl/index.html
Scraping: https://pantelis.github.io/book/vla/index.html
Scraping: https://pantelis.github.io/courses/ai/index.html
Scraping: https://pantelis.github.io/courses/robotics/index.html
Scraping: 